# Linking citizen-science observations to TESSERA embeddings — feasibility check

**Purpose.** First, minimal check of whether the two pilot datasets
(`../02_citizen_science/`, `../03_satellite/`) can actually be joined:
sample the 128-d TESSERA embedding at each observation's location. This is
**not** dataset construction — no temporal matching, no filtering, no
negative sampling. All of that is still open (see
`../01_problem_exploration/available_data_exploration.ipynb` and
`docs/research_questions.md`, Shared Design).

**Scope.** Pilot data only (see `data/README.md`): iNaturalist genus
*Acacia* (Coimbra district) and TESSERA 2024 (Coimbra municipality) — two
different boundaries, reconciled here only by intersection, not by
redefining either pilot's scope.

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from shapely.geometry import box

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"
INAT_CSV = RAW / "citizen_science" / "inaturalist_acacia_coimbra_pilot.csv"
TESSERA_DIR = RAW / "satellite" / "tessera_coimbra_pilot"
TESSERA_YEAR = 2024
TIFF_DIR = TESSERA_DIR / "global_0.1_degree_tiff_all"
UTM = "EPSG:32629"

inat = pd.read_csv(INAT_CSV)
inat["observed_on"] = pd.to_datetime(inat["observed_on"], errors="coerce")
inat_gdf = gpd.GeoDataFrame(
    inat, geometry=gpd.points_from_xy(inat["longitude"], inat["latitude"]), crs="EPSG:4326"
).to_crs(UTM)

tile_dirs = sorted((TESSERA_DIR / "global_0.1_degree_representation" / str(TESSERA_YEAR)).iterdir())
tile_centres = [tuple(float(v) for v in d.name.removeprefix("grid_").split("_")) for d in tile_dirs]
tiles = gpd.GeoDataFrame(
    {"tile": [d.name for d in tile_dirs]},
    geometry=[box(lon - 0.05, lat - 0.05, lon + 0.05, lat + 0.05) for lon, lat in tile_centres],
    crs="EPSG:4326",
).to_crs(UTM)
inat_gdf["in_tessera"] = inat_gdf.within(tiles.union_all())

pd.set_option("display.max_columns", 50)

## 1. Sampling embeddings at observation locations

Sample the 128-d embedding at each observation inside the downloaded tiles
(nearest pixel). This is a feasibility check, not dataset construction: no
filtering, no temporal matching (all records get the 2024 embedding
regardless of their observation year).

In [ ]:
def sample_tile(tile_dir, points):
    with rasterio.open(TIFF_DIR / f"{tile_dir.name}.tiff") as src:
        rows, cols = rasterio.transform.rowcol(src.transform, points.geometry.x, points.geometry.y)
        h, w = src.shape
    rows, cols = np.asarray(rows), np.asarray(cols)
    ok = (rows >= 0) & (rows < h) & (cols >= 0) & (cols < w)
    emb = np.load(tile_dir / f"{tile_dir.name}.npy", mmap_mode="r")
    scales = np.load(tile_dir / f"{tile_dir.name}_scales.npy", mmap_mode="r")
    vecs = emb[rows[ok], cols[ok]].astype(np.float32) * scales[rows[ok], cols[ok]][:, None]
    return points.index[ok], vecs

pts = inat_gdf[inat_gdf["in_tessera"]]
tile_of_point = gpd.sjoin(pts[["geometry"]], tiles, predicate="within")["tile"]
idx, vecs = [], []
for d in tile_dirs:
    sub = pts.loc[tile_of_point.index[tile_of_point == d.name]]
    if len(sub):
        i, v = sample_tile(d, sub)
        idx.extend(i); vecs.append(v)
point_embeddings = pd.DataFrame(np.vstack(vecs), index=idx).sort_index()
print(f"Embeddings sampled for {len(point_embeddings):,} observations "
      f"({point_embeddings.round(4).drop_duplicates().shape[0]:,} distinct vectors — duplicates = same pixel)")
print("Observation years of these records:")
print(inat["observed_on"].dt.year.loc[point_embeddings.index].value_counts().sort_index().to_dict())

## Summary

- 928 of 7,943 observations (11.7%) fall inside the downloaded TESSERA
  tiles — the rest have no embedding, purely because the two pilot
  boundaries don't match (see `data/README.md`).
- That 928-observation subset skews recent (mostly 2023–2026) relative to
  the full dataset's 2021 peak — so results on this subset would not
  necessarily transfer to the rest of the data.
- Every matched observation gets the same 2024 embedding regardless of its
  own observation year — no temporal matching is applied here. The
  proposed rule (2023 reference year, 2021–2025 window) is in
  `docs/research_questions.md`, Temporal Design.

This is a feasibility check only — actual dataset construction (temporal
matching, negative sampling, train/test splits) is defined in
`docs/research_questions.md` and `docs/methodology.md`.